# C3: spaCy ili SBERT po izboru + aglomerativno klasterovanje (Amazon komentari)
**Tip:** Blok 4, zadatak za vežbu. Reprezentaciju biraš sam i obrazlažeš.

### Zadatak
Fajl `Amazon_Reviews_for_class.csv` sadrži komentare proizvoda sa Amazona. Koristeći kolone `Summary` i `Text`, izvršiti klasterizaciju komentara **aglomerativnim hijerarhijskim klasterovanjem**.
- Za vektorsku reprezentaciju koristiti **ili vektore reči iz spaCy modela, ili SBERT**. Obrazložiti izbor.
- Broj klastera na osnovu dendrograma i siluete, obrazložiti.
- Top 10 reči po TF-IDF za svaki klaster, veličina i opis klastera.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import seaborn as sb
from pypalettes import load_cmap

from pathlib import Path
from IPython.display import display

import spacy
from sentence_transformers import SentenceTransformer

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'
MODELS_DIR = Path.cwd().parent / 'models'

### Šta je urađeno
- Učitani podaci (4000 komentara), provereni prazni i duplikati.
- Prazan `Summary` zamenjen praznim stringom, napravljena `content` (Summary + tačka + Text).
- Dužina: medijana oko 60 reči, ali ima i dugih komentara (do oko 1500 reči). Za spaCy to nije problem, a SBERT sam odseca tekst posle oko 100 reči (128 tokena), što možeš da pomeneš u obrazloženju izbora.
- `Score` (ocena) i `ProductId` ne ulaze u klasterizaciju, ali mogu da pomognu pri opisu klastera.

In [ ]:
data = pd.read_csv(DATA_DIR / 'Amazon_Reviews_for_class.csv')
data.info()

In [ ]:
print(data[['Summary', 'Text']].isna().sum())
data = data.dropna(subset=['Text'])
print('Duplikati teksta:', data.duplicated(subset='Text').sum())
data = data.drop_duplicates(subset='Text').reset_index(drop=True)

data['Summary'] = data['Summary'].fillna('')
data['content'] = data['Summary'] + '. ' + data['Text']

In [ ]:
print(data['content'].str.split().str.len().describe())

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. Napiši u markdown-u koju reprezentaciju biraš i zašto (npr. kratki komentari i kontekst idu SBERT-u; spaCy je brži)
2. Vektori (spaCy: čišćenje i prosek, 300 dim.; SBERT: sirov tekst, `normalize_embeddings=True`, 384 dim.)
3. Linkage, dendrogram, silueta, izbor broja klastera
4. Ključne reči po klasteru, grafik, opis klastera (može i `data.groupby('clust')['Score'].mean()`)

---
## Dati kod
Ovo dobijaš uz zadatak.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): prikaz kljucnih reci po klasterima
# Funkciju get_cluster_keywords(cluster_id) pises sam; treba da vrati listu parova (rec, skor).
# Raskomentarisi kad stignes do ovog koraka.

# fig, ax_grid = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), constrained_layout=True)
# axes = ax_grid.flatten()
# cmap = load_cmap("excel_Median")
#
# n_clust = data['clust'].nunique()
# for i in range(n_clust):
#     scored_kwds = get_cluster_keywords(i+1)
#     scored_kwds = sorted(scored_kwds, key=lambda item: item[1])
#     scores = [s for kw, s in scored_kwds]
#     keywords = [kw for kw, s in scored_kwds]
#     axes[i].barh(keywords, scores, color=cmap(i))
#     axes[i].set_title(f"Cluster {i+1}", color=cmap(i))
#     axes[i].grid(visible=True, axis='y', color='gray', alpha=0.35)
#
# for ax in axes[n_clust:]:   # brise prazna polja ako je mreza veca od broja klastera
#     ax.remove()
#
# fig.suptitle("Kljucne reci po klasterima")
# plt.show()

In [ ]:
# DATI KOD (primer): skraceni dendrogram, ako je pun nepregledan
# plt.figure(figsize=(12, 6))
# sch.dendrogram(ward_linkage, truncate_mode='lastp', p=30)
# plt.title('Dendrogram (Ward)')
# plt.ylabel('Distanca')
# plt.show()